# 🏋️ ระบบสร้าง Reference Pose สำหรับ AI Exercise System

Notebook นี้ใช้สำหรับสร้างข้อมูล Landmark ต้นแบบของแต่ละท่าออกกำลังกาย
โดยใช้ MediaPipe Pose ดึง Landmark จากรูปภาพ แล้วบันทึกเป็น JSON

**ท่าที่รองรับ:**
- Squat
- Push Up
- Sit Up
- Plank
- Lunge
- Jumping Jack
- Bicep Curl
- Shoulder Press
- Mountain Climber
- Burpee

In [ ]:
!pip install mediapipe opencv-python-headless numpy

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import json
import os
import math
from google.colab import files
import matplotlib.pyplot as plt

mp_pose = mp.solutions.pose
mp_draw = mp.solutions.drawing_utils

print('✅ Libraries loaded successfully')

In [ ]:
def calculate_angle(p1, p2, p3):
    """Calculate angle between three points"""
    x1, y1 = p1
    x2, y2 = p2
    x3, y3 = p3
    angle = math.degrees(math.atan2(y3 - y2, x3 - x2) - math.atan2(y1 - y2, x1 - x2))
    if angle < 0:
        angle += 360
    if angle > 180:
        angle = 360 - angle
    return angle

def extract_pose_data(image_path):
    """Extract pose landmarks and angles from an image"""
    img = cv2.imread(image_path)
    if img is None:
        print(f'❌ Cannot read image: {image_path}')
        return None
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    with mp_pose.Pose(static_image_mode=True, model_complexity=2, min_detection_confidence=0.5) as pose:
        results = pose.process(img_rgb)
        
        if not results.pose_landmarks:
            print(f'❌ No pose detected in: {image_path}')
            return None
        
        landmarks = []
        for lm in results.pose_landmarks.landmark:
            landmarks.append({'x': lm.x, 'y': lm.y, 'z': lm.z, 'visibility': lm.visibility})
        
        def get_lm(idx):
            return (landmarks[idx]['x'], landmarks[idx]['y'])
        
        angles = {
            'left_elbow': calculate_angle(get_lm(11), get_lm(13), get_lm(15)),
            'right_elbow': calculate_angle(get_lm(12), get_lm(14), get_lm(16)),
            'left_shoulder': calculate_angle(get_lm(13), get_lm(11), get_lm(23)),
            'right_shoulder': calculate_angle(get_lm(14), get_lm(12), get_lm(24)),
            'left_hip': calculate_angle(get_lm(11), get_lm(23), get_lm(25)),
            'right_hip': calculate_angle(get_lm(12), get_lm(24), get_lm(26)),
            'left_knee': calculate_angle(get_lm(23), get_lm(25), get_lm(27)),
            'right_knee': calculate_angle(get_lm(24), get_lm(26), get_lm(28)),
        }
        
        # Draw landmarks on image for visualization
        annotated = img_rgb.copy()
        mp_draw.draw_landmarks(annotated, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        
        return {
            'landmarks': landmarks,
            'angles': angles,
            'image': annotated
        }

print('✅ Functions defined')

In [ ]:
# Upload exercise images
print('📤 กรุณาอัพโหลดรูปภาพท่าออกกำลังกาย')
uploaded = files.upload()

In [ ]:
# Process each uploaded image
os.makedirs('reference_pose', exist_ok=True)

# Map filename to exercise name
# Example: squat_front.jpg -> exercise=squat, view=front
results_data = {}

for filename in uploaded.keys():
    parts = filename.replace('.jpg','').replace('.png','').split('_')
    if len(parts) >= 2:
        exercise = parts[0]
        view = '_'.join(parts[1:])
    else:
        exercise = parts[0]
        view = 'front'
    
    print(f'\n🔍 Processing: {filename} -> Exercise: {exercise}, View: {view}')
    
    data = extract_pose_data(filename)
    if data:
        # Save JSON
        os.makedirs(f'reference_pose/{exercise}', exist_ok=True)
        json_path = f'reference_pose/{exercise}/{view}.json'
        
        save_data = {
            'exercise': exercise,
            'view': view,
            'landmarks': data['landmarks'],
            'angles': data['angles']
        }
        
        with open(json_path, 'w') as f:
            json.dump(save_data, f, indent=2)
        
        print(f'✅ Saved: {json_path}')
        print(f'   Angles: {data["angles"]}')
        
        # Display image with landmarks
        plt.figure(figsize=(8, 6))
        plt.imshow(data['image'])
        plt.title(f'{exercise} - {view}')
        plt.axis('off')
        plt.show()
        
        results_data[f'{exercise}_{view}'] = save_data
    else:
        print(f'❌ Failed to process: {filename}')

In [ ]:
# Create zip of all reference poses and download
import zipfile

with zipfile.ZipFile('reference_pose.zip', 'w') as zipf:
    for root, dirs, files_list in os.walk('reference_pose'):
        for file in files_list:
            zipf.write(os.path.join(root, file))

print('📦 Created reference_pose.zip')
files.download('reference_pose.zip')
print('✅ Download started!')